# Extração de Dados SRAG

Download e verificação do Parquet canônico `dados-v2/srag.parquet`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


ROOT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from src import config as env
from src import config as C
from src import dados as storage
from src.dados import get_parquet


env.carregar_dotenv()
env.garantir_diretorios()

print("ROOT_DIR:", ROOT_DIR)
print(env.resumo_ambiente())

In [ ]:
print(f"Repositório:  {C.GITHUB_REPO}")
print(f"Versão:       {C.DATA_VERSION}")
print(f"Arquivo:      {C.PARQUET_NAME}")
print(f"URL canônica: {C.srag_parquet_url()}")

In [ ]:
parquet_path = get_parquet()

print(f"\nParquet em: {parquet_path}")
print(f"Tamanho:    {parquet_path.stat().st_size / 1024**2:.2f} MB")

In [ ]:
import time
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

inicio=time.perf_counter()
arquivo=pq.ParquetFile(parquet_path)
meta=arquivo.metadata
quantidade_registros=int(meta.num_rows)
quantidade_colunas=int(meta.num_columns)
quantidade_grupos_linhas=int(meta.num_row_groups)
tamanho_arquivo_mb=parquet_path.stat().st_size/1_048_576

if quantidade_registros<=0 or quantidade_colunas<=0:
    raise ValueError("Parquet vazio ou sem colunas.")

lote=next(arquivo.iter_batches(batch_size=5),None)
if lote is None:
    raise ValueError("Não foi possível extrair a amostra.")
amostra_parquet=lote.to_pandas(types_mapper=pd.ArrowDtype).reset_index(drop=True)
memoria_amostra_mb=amostra_parquet.memory_usage(index=True,deep=True).sum()/1_048_576

resumo_ingestao=pd.DataFrame({"metrica":["caminho","tamanho_mb","registros","colunas","row_groups","linhas_amostra","memoria_amostra_mb","tempo_s"],
"valor":[str(parquet_path),round(tamanho_arquivo_mb,2),quantidade_registros,quantidade_colunas,quantidade_grupos_linhas,len(amostra_parquet),round(memoria_amostra_mb,4),round(time.perf_counter()-inicio,3)]})
display(amostra_parquet)
display(resumo_ingestao)
print("Verificação concluída sem carregar o dataset completo.")